# Experiment Idn: Coherent-RE
* Sample Idn: [80TeZn2Er0Sm](https://www.sciencedirect.com/science/article/pii/S0009261419309522)

* Modulado pelo AOM (Driver: 28.5 V, consome 0.54 A).
Amplitude de modulação (Gerador de função):
  * offset: 0.500 V
  * amplitudes: { 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50 }

* Corrente fornecida pelo driver para o laser: 750 mA

Contagens (room T):
* Full bands: ~?? couts/100ms

Longer acquisition times may lead to increased contribution of laser leackage ?

## Configurações

In [1]:
# --- Célula de Importação ---
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy import signal as sig
from scipy.optimize import curve_fit
from scipy.integrate import odeint, trapezoid
from scipy.fft import fft, fftfreq

# Substituindo widgets do Colab por ipywidgets padrão do Jupyter
import ipywidgets as widgets
from IPython.display import display, clear_output

np.seterr('warn')

kB = 0.695034 #cm^-1 / K  #Boltzmann's Constant
hc = 1.98644568e-25 # J.m  #Planck's Constant

In [2]:
# --- Configuração do Caminho (Ajuste aqui!) ---
# No Linux/Ubuntu, use o caminho absoluto ou relativo da sua pasta local
# Exemplo: data_path = './dados_experimento/'
samples = ["y2o4_yb_er3+", "y2o4_er3+", "nayf4_yb_er3+"]

data_path = './experimental_data'
freq_list = [1, 10, 100, 1000, 10000]
amp_list = list(range(5, 55,5))
print(amp_list)

[5, 10, 15, 20, 25, 30, 35, 40, 45, 50]


In [3]:
# --- Carregamento de Arquivos ---
filenames_all = []

for root, dirs, files in os.walk(data_path):
    # print(root)
    for name in files:
        # os.path.join combina o diretório atual (root) com o nome do arquivo
        # print(name)
        full_path = os.path.join(root, name)
        filenames_all.append(full_path)

# print(filenames_all)


In [4]:
def format_single_path(data_paths, freq=None, amp=None, alisson_data=False):
    if alisson_data:
        filtered_paths = [fn for fn in data_paths if ((f'_{freq}Hz_' in fn) and (f"{(amp*1e-2):.2f}amp" in fn))]
    else:
        filtered_paths = [fn for fn in data_paths if ((f'{freq}hz.txt' in fn) and (f"amp{amp}e-2_" in fn))]
    
    if len(filtered_paths) != 1:
        print("Array não unitário: ", filtered_paths)
        # raise ValueError("Not unitary array")
        return None
    else: 
        return filtered_paths 

def format_data(data_paths: list[str], allison_data=False) -> list:
    result = []
    
    for amp in amp_list:
        # print(amp)
        data = {freq: np.loadtxt(
            format_single_path(data_paths, freq=freq, amp=amp, alisson_data=allison_data)
            ) for freq in freq_list}
        result.append({
            "amp": amp*1e-2,
            "data": data
            })
        
    return result  

In [ ]:
np.

In [5]:
laser_paths = [fn for fn in filenames_all if ("laser" in fn)]
laser_data = format_data(laser_paths, allison_data=True) 

ValueError: could not convert string './experimental_data\\06-04-26\\laser_1Hz_750mA_0.5offset_0.05amp_10rep_GPTIM.txt' to float64 at row 0, column 1.

In [ ]:
sample1_paths = [fn for fn in filenames_all if ("y2o3_yb_er" in fn)]
sample1_data = format_data(sample1_paths)

In [ ]:
sample2_paths = [fn for fn in filenames_all if ("y2o3_er" in fn)]
[print(fn) for fn in sample2_paths]
# sample2_data = format_data(sample2_paths)

# sample3_paths = [fn for fn in filenames_all if ("nayf4_yb_er" in fn)]
# sample3_data = format_data(sample3_paths)

In [ ]:
# --- Função de FFT (Mantida igual) ---
def get_fft(freq, time, curve):
    sample_interval = max(time)/len(curve)/1e6  # seconds per sample
    N = len(curve)
    yf = fft(curve)
    xf = fftfreq(N, sample_interval)
    return xf, yf

# Processamento dos dicionários (Mantido igual)
laser_GPTIM = {
    float(
        fn.split('_')[1].replace('Hz', '')
        ): np.loadtxt(os.path.join(folder_to_scan, fn)) for fn in filenames_all if ('laser' in fn and 'GPTIM' in fn)}
laser_LRTIM = {float(fn.split('_')[1].replace('Hz', '')): np.loadtxt(os.path.join(folder_to_scan, fn)) for fn in filenames_all if ('laser' in fn and 'LRTIM' in fn)}
verdes_GPTIM = {float(fn.split('_')[1].replace('Hz', '')): np.loadtxt(os.path.join(folder_to_scan, fn)) for fn in filenames_all if ('verdes' in fn and 'GPTIM' in fn)}
verdes_LRTIM = {float(fn.split('_')[1].replace('Hz', '')): np.loadtxt(os.path.join(folder_to_scan, fn)) for fn in filenames_all if ('verdes' in fn and 'LRTIM' in fn)}

## Plots

In [ ]:
# --- Visualização com Interatividade Local (Substituindo TabBar) ---
def plot_data(freq):
    time = laser_LRTIM[freq][:, 0]
    laser_curve = laser_LRTIM[freq][:, 1]
    verdes_curve = verdes_LRTIM[freq][:, 1]

    fig, ax = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    
    # Plot no tempo
    ax[0].plot(time, (laser_curve-min(laser_curve))/(max(laser_curve)-min(laser_curve)), label='Laser', color='red')
    ax[0].plot(time, (verdes_curve-min(verdes_curve))/(max(verdes_curve)-min(verdes_curve)), label='Verdes', color='green')
    ax[0].set_xlabel('Time (us)')
    ax[0].set_ylabel('Normalized Intensity')
    ax[0].legend()

    # Plot FFT
    xf_laser, yf_laser = get_fft(freq, time, laser_curve)
    xf_verdes, yf_verdes = get_fft(freq, time, verdes_curve)
    N = len(yf_laser)//2
    
    ax[1].plot(xf_laser[1:N], np.abs(yf_laser[1:N]) / N * 2, label='Laser', color='red')
    ax2 = ax[1].twinx()
    ax2.plot(xf_verdes[1:N], np.abs(yf_verdes[1:N]) / N * 2, label='Verdes', color='green')
    
    ax[1].set_xlabel('Frequency (Hz)')
    ax[1].set_ylabel('FFT')
    ax[1].set_xscale('log')
    fig.suptitle(f'LRTIM - Frequency: {freq} Hz')
    plt.show()

# Criando um menu dropdown (mais estável no Jupyter local que abas CSS)
freq_options = sorted(verdes_LRTIM.keys())
widgets.interact(plot_data, freq=freq_options);